<a href="https://colab.research.google.com/github/Burakcelebi2003/RE24-Baseball-Sabermetrik-Predict/blob/main/RE24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = ['RndmForst', 'LineerR.', 'AR(1)', 'Ensemble', 'BootstrapCmmprt', 'EWMA', 'PolinomTrent']
values1 = [7, 7, 7, -5, 7, 9, 7]
values2 = [11, 5, 6, 6, 7, 10, 7]
x = np.arange(len(categories))
width = 0.4

plt.figure(figsize=(12, 6))
bars1 = plt.bar(x - width/2, values1, width, label='Mariners', color='#005C5C', edgecolor='black')
bars2 = plt.bar(x + width/2, values2, width, label='Blue Jays', color='#134A8E', edgecolor='black')

plt.title('Game 7', fontsize=16)
plt.xlabel('Metrics', fontsize=14)
plt.ylabel('Values', fontsize=14)
plt.xticks(x, categories)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

re24_real = [
    1.05, 0.62, 0.21, -0.01, 1.16, -0.1, -0.55, 2.23, -0.98, 0.04,
    -0.17, -0.11, -0.24, -0.74, -0.40, 1.30, 1.29, 1.69, 1.38, 2.51, 0.53, 1.39, 0.41, 0.34, 0.06, 2.05, 0.61
]

X, y = [], []
for i in range(3, len(re24_real)):
    features = [
        re24_real[i-1],
        re24_real[i-2],
        re24_real[i-3],
        np.mean(re24_real[i-3:i])
    ]
    X.append(features)
    y.append(re24_real[i])

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

simulated_re24s = re24_real.copy()
for i in range(len(re24_real), 162):
    last3 = simulated_re24s[-3:]
    test_input = [
        last3[2], last3[1], last3[0],
        np.mean(last3)
    ]
    predicted_value = model.predict([test_input])[0]
    noise = np.random.normal(0, 0.8)  # Doğal sapma
    simulated_value = predicted_value + noise
    simulated_re24s.append(simulated_value)

results = []
for i in range(28, 111):
    value = simulated_re24s[i-1]
    durum = "✅" if value > 0 else "❌"
    results.append((i, value, durum))

values_only = [v for _, v, _ in results]
mean_value = np.mean(values_only)
variance_value = np.var(values_only)

for i, value, durum in results:
    print(f"{i:3}. maç RE24 tahmini: {value:+.2f} → {durum}")

print("\n    Sezon Sonu Results    ")
print(f"Ortalama: {mean_value:.3f}")
print(f"Varyans: {variance_value:.3f}")
print(f"0+: {sum(1 for v in values_only if v > 0)}")
print(f"0-: {sum(1 for v in values_only if v <= 0)}")

In [ ]:
#Ronnie (Ronad Acuna stats)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

re24_real = [
     -1.46, 0.95, -0.36, 0.73, 1.13, -1.71, -0.60, -1.51, -0.15, 1.75
]

X, y = [], []
for i in range(3, len(re24_real)):
    features = [
        re24_real[i-1],
        re24_real[i-2],
        re24_real[i-3],
        np.mean(re24_real[i-3:i])
    ]
    X.append(features)
    y.append(re24_real[i])

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

simulated_re24s = re24_real.copy()
for i in range(len(re24_real), 162):
    last3 = simulated_re24s[-3:]
    test_input = [
        last3[2], last3[1], last3[0],
        np.mean(last3)
    ]
    predicted_value = model.predict([test_input])[0]
    noise = np.random.normal(0, 0.8)
    simulated_value = predicted_value + noise
    simulated_re24s.append(simulated_value)

results = []
for i in range(1, 12):
    value = simulated_re24s[i-1]
    durum = "✅" if value > 0 else "❌"
    results.append((i, value, durum))

match_numbers = [i for i, _, _ in results]
re24_values = [v for _, v, _ in results]

positive_count = sum(1 for v in re24_values if v > 0)
negative_count = sum(1 for v in re24_values if v <= 0)

plt.figure(figsize=(14, 7))
plt.scatter(
    match_numbers, re24_values,
    c=['green' if v > 0 else 'red' for v in re24_values],
    alpha=0.7
)

for x, y_val in zip(match_numbers, re24_values):
    plt.text(x, y_val, f"{y_val:.2f}", fontsize=7, ha='center', va='bottom')

plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title(f"Ronald 2025 sezonu RE24 Tahmini\n0+: {positive_count} maç / 0-: {negative_count} maç")
plt.xlabel("Maç")
plt.ylabel("RE24 Tahmini")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures

son_10_mac = [3.96, -1.12, 0.36, -0.21, 0.44, -0.33, -1.25, 0.26, 0.17, -0.67]

def moving_average(data, window=3):
    return [np.mean(data[i-window:i]) for i in range(window, len(data)+1)]

def polynomial_trend(data, degree=2):
    x = np.arange(len(data))
    coeffs = np.polyfit(x, data, degree)
    return np.polyval(coeffs, len(data))

def weighted_average(data, weights=[0.5, 0.3, 0.2]):
    return np.dot(data[-len(weights):], weights)

X, y = [], []
for i in range(3, len(son_10_mac)):
    features = [
        son_10_mac[i-1],
        son_10_mac[i-2],
        son_10_mac[i-3],
        np.mean(son_10_mac[i-3:i]),
        np.std(son_10_mac[i-3:i]),
        polynomial_trend(son_10_mac[:i]),
        weighted_average(son_10_mac[:i])
    ]
    X.append(features)
    y.append(son_10_mac[i])

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
    random_state=42
)
model.fit(X, y)

son_3_mac = son_10_mac[-3:]
features_11 = [
    son_10_mac[-1],
    son_10_mac[-2],
    son_10_mac[-3],
    np.mean(son_3_mac),
    np.std(son_3_mac),
    polynomial_trend(son_10_mac),
    weighted_average(son_10_mac)
]

tahmini_re24 = model.predict([features_11])[0]
gurultu = np.random.normal(0, 0.15)
tahmini_re24 += gurultu

tum_maclar = son_10_mac + [tahmini_re24]
sonuclar = []
for i, deger in enumerate(tum_maclar, 1):
    durum = "✅" if deger > 0 else "❌"
    sonuclar.append((i, deger, durum))

print("╔══════════════════════════════╗")
print("║        DETAYLI ANALİZ        ║")
print("╠════════╦══════════╦══════════╣")
print("║ Maç No ║ RE24     ║ Durum    ║")
print("╠════════╬══════════╬══════════╣")
for mac, deger, durum in sonuclar:
    print(f"║ {mac:6d} ║ {deger:7.2f}║ {durum:7s}   ║")
print("╚════════╩══════════╩══════════╝")

print("\nHESAPLAMA TEKNİKLERİ:")
print(f"- 3'lü Hareketli Ortalama: {np.mean(son_3_mac):.2f}")
print(f"- Standart Sapma: {np.std(son_3_mac):.2f}")
print(f"- Polinom Trend Tahmini: {polynomial_trend(son_10_mac):.2f}")
print(f"- Ağırlıklı Ortalama: {weighted_average(son_10_mac):.2f}")
print(f"- Model Tahmini: {tahmini_re24-gurultu:.2f} ± {abs(gurultu):.2f}")

plt.figure(figsize=(12, 6))
plt.plot(range(1, 12), tum_maclar, 'o-', color='blue', markersize=8)
plt.axhline(0, color='red', linestyle='--')
plt.title("11 Maçlık RE24 Performansı (10 Gerçek + 1 Tahmin)", pad=20)
plt.xlabel("Maç No", labelpad=10)
plt.ylabel("RE24 Değeri", labelpad=10)
plt.grid(True, alpha=0.3)

plt.plot(11, tahmini_re24, 'o', markersize=10, markerfacecolor='none',
         markeredgecolor='green', markeredgewidth=2)
plt.text(11, tahmini_re24, f" {tahmini_re24:.2f}", ha='left', va='center')

plt.xticks(range(1, 12))
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
from scipy.stats import skew, kurtosis, zscore
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

son_10_mac = [2.64, -0.87, 2.65, 1.59, -0.87, 2.69, -0.48, -0.45, 1.44, -1.11]

def ewma(data, span=3):
    return pd.Series(data).ewm(span=span, adjust=False).mean().iloc[-1]

def moving_average(data, window=3):
    return [np.mean(data[i-window:i]) for i in range(window, len(data)+1)]

def polynomial_trend(data, degree=2, predict_at=None):
    x = np.arange(len(data))
    coeffs = np.polyfit(x, data, degree)
    if predict_at is None:
        predict_at = len(data)
    return np.polyval(coeffs, predict_at)

def weighted_average(data, weights=[0.5, 0.3, 0.2]):
    wlen = len(weights)
    vec = np.array(data[-wlen:])
    return float(np.dot(vec, np.array(weights)))

def rolling_features(series, window):
    s = pd.Series(series)
    return {
        f"roll_mean_{window}": s.rolling(window).mean().iloc[-1],
        f"roll_med_{window}": s.rolling(window).median().iloc[-1],
        f"roll_std_{window}": s.rolling(window).std().iloc[-1],
        f"roll_skew_{window}": s.rolling(window).apply(lambda x: skew(x), raw=True).iloc[-1],
        f"roll_kurt_{window}": s.rolling(window).apply(lambda x: kurtosis(x), raw=True).iloc[-1],
    }

X, y = [], []
for i in range(7, len(son_10_mac)):
    window3 = son_10_mac[i-3:i]
    window5 = son_10_mac[i-5:i]
    features = {
        "lag_1": son_10_mac[i-1],
        "lag_2": son_10_mac[i-2],
        "lag_3": son_10_mac[i-3],
        "lag_4": son_10_mac[i-4],
        "lag_5": son_10_mac[i-5],
        "lag_6": son_10_mac[i-6],
        "lag_7": son_10_mac[i-7],
        "mean_3": np.mean(window3),
        "std_3": np.std(window3, ddof=1),
        "mean_5": np.mean(window5),
        "std_5": np.std(window5, ddof=1),
        "ewma_3": ewma(son_10_mac[:i], span=3),
        "poly_trend": polynomial_trend(son_10_mac[:i], degree=2, predict_at=len(son_10_mac[:i])),
        "wavg_3": weighted_average(son_10_mac[:i]),
        **rolling_features(son_10_mac[:i], 5)
    }
    X.append(list(features.values()))
    y.append(son_10_mac[i])

feature_names = list(features.keys())

X = np.array(X)
y = np.array(y)

if X.shape[0] < 3:
    print("Uyarı: Çok az eğitim örneği var. Modellerin performansı sınırlı olacaktır.")

rf = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)
lr = LinearRegression()

rf.fit(X, y)
lr.fit(X, y)

importances = rf.feature_importances_
feat_imp = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)

son_3_mac = son_10_mac[-3:]
son_5_mac = son_10_mac[-5:]
feature_11 = {
    "lag_1": son_10_mac[-1],
    "lag_2": son_10_mac[-2],
    "lag_3": son_10_mac[-3],
    "lag_4": son_10_mac[-4],
    "lag_5": son_10_mac[-5],
    "lag_6": son_10_mac[-6],
    "lag_7": son_10_mac[-7],
    "mean_3": np.mean(son_3_mac),
    "std_3": np.std(son_3_mac, ddof=1),
    "mean_5": np.mean(son_5_mac),
    "std_5": np.std(son_5_mac, ddof=1),
    "ewma_3": ewma(son_10_mac, span=3),
    "poly_trend": polynomial_trend(son_10_mac, degree=2, predict_at=len(son_10_mac)),
    "wavg_3": weighted_average(son_10_mac),
    **rolling_features(son_10_mac, 5)
}

X11 = np.array(list(feature_11.values())).reshape(1, -1)

pred_rf = rf.predict(X11)[0]
pred_lr = lr.predict(X11)[0]

arima_pred = None
try:
    series = pd.Series(son_10_mac)
    model_arima = ARIMA(series, order=(1,0,0)).fit()
    steps = 1
    arima_fore = model_arima.forecast(steps=steps)
    arima_pred = float(arima_fore.iloc[-1])
except Exception as e:
    arima_pred = np.nan

preds = [pred_rf, pred_lr]
if not np.isnan(arima_pred):
    preds.append(arima_pred)
ensemble_pred = np.nanmean(preds)

n_boot = 1000
boot_preds = []
rng = np.random.RandomState(42)
for _ in range(n_boot):

    idx = rng.choice(len(X), size=len(X), replace=True)
    Xb, yb = X[idx], y[idx]
    m = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=rng.randint(1e6))
    try:
        m.fit(Xb, yb)
        p = m.predict(X11)[0]
        boot_preds.append(p)
    except Exception:
        continue

if len(boot_preds) > 0:
    lower, upper = np.percentile(boot_preds, [2.5, 97.5])
else:
    lower, upper = ensemble_pred - 0.8, ensemble_pred + 0.8

simulated = []
for noise_sd in [0.05, 0.1, 0.2]:
    simulated.append({
        "noise_sd": noise_sd,
        "rf_pred_with_noise": pred_rf + np.random.normal(0, noise_sd),
        "ensemble_with_noise": ensemble_pred + np.random.normal(0, noise_sd)
    })

tum_maclar = son_10_mac + [ensemble_pred]
sonuclar = []
for i, deger in enumerate(tum_maclar, 1):
    durum = "✅" if deger > 0 else "❌"
    sonuclar.append((i, deger, durum))

print("╔══════════════════════════════╗")
print("║        DETAYLI ANALİZ        ║")
print("╠════════╦══════════╦══════════╣")
print("║ Maç No ║ RE24     ║ Durum    ║")
print("╠════════╬══════════╬══════════╣")
for mac, deger, durum in sonuclar:
    print(f"║ {mac:6d} ║ {deger:7.2f}║ {durum:7s}   ║")
print("╚════════╩══════════╩══════════╝\n")

print("MODELLER ve TAHMİNLER:")
print(f"- RandomForest tahmini: {pred_rf:.3f}")
print(f"- LinearRegression tahmini: {pred_lr:.3f}")
if not np.isnan(arima_pred):
    print(f"- ARIMA(1,0,0) tahmini: {arima_pred:.3f}")
print(f"- Ensemble (ortalama) tahmini: {ensemble_pred:.3f}")
print(f"- %95 Tahmin Aralığı (bootstrap): [{lower:.3f}, {upper:.3f}]\n")

print("EK ÖZETLER ve ÖZELLİK ÖNEMLERİ:")
print(f"- Son 3 maç ortalaması: {np.mean(son_3_mac):.3f}")
print(f"- Son 5 maç ortalaması: {np.mean(son_5_mac):.3f}")
print(f"- EWMA(3): {ewma(son_10_mac, span=3):.3f}")
print(f"- Polinom trend (deg2) tahmini: {polynomial_trend(son_10_mac):.3f}")
print("\n- Feature importances (RandomForest):")
for feat, imp in feat_imp:
    print(f"  {feat:20s}: {imp:.3f}")

print("\n- Gürültü duyarlılık örnekleri (rf ve ensemble):")
for s in simulated:
    print(f"  noise_sd={s['noise_sd']}: rf={s['rf_pred_with_noise']:.3f}, ensemble={s['ensemble_with_noise']:.3f}")

plt.figure(figsize=(12, 6))
plt.plot(range(1, 12), tum_maclar, 'o-', color='tab:blue', markersize=8, label='Gerçek + Tahmin')
plt.axhline(0, color='red', linestyle='--', label='Eşik 0')
plt.title("11 Maçlık RE24 Performansı (10 Gerçek + 1 Ensemble Tahmin)", pad=14)
plt.xlabel("Maç No")
plt.ylabel("RE24 Değeri")
plt.grid(True, alpha=0.3)

plt.plot(11, ensemble_pred, 's', markersize=10, markerfacecolor='gold', markeredgecolor='black', label='Ensemble Tahmin')
plt.text(11.05, ensemble_pred, f" {ensemble_pred:.2f}", va='center')

plt.fill_between([10.5, 11.5], lower, upper, color='gold', alpha=0.2, label='%95 Tahmin Aralığı (bootstrap)')

plt.xticks(range(1, 12))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression as PolyLR
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

re24_real = [0.13, 0.48, ]
wpa_real = [-0.002, 0.105, ]

X_re24, y_re24 = [], []
X_wpa, y_wpa = [], []

for i in range(3, len(re24_real)):

    features_re24 = [re24_real[i-1], re24_real[i-2], re24_real[i-3], np.mean(re24_real[i-3:i])]
    X_re24.append(features_re24)
    y_re24.append(re24_real[i])


    features_wpa = [wpa_real[i-1], wpa_real[i-2], wpa_real[i-3], np.mean(wpa_real[i-3:i])]
    X_wpa.append(features_wpa)
    y_wpa.append(wpa_real[i])

next_match_idx = len(re24_real)


rf_re24 = RandomForestRegressor(random_state=42).fit(X_re24, y_re24)
rf_re24_pred = rf_re24.predict([X_re24[-1]])[0]

rf_wpa = RandomForestRegressor(random_state=42).fit(X_wpa, y_wpa)
rf_wpa_pred = rf_wpa.predict([X_wpa[-1]])[0]

lr_re24 = LinearRegression().fit(X_re24, y_re24)
lr_re24_pred = lr_re24.predict([X_re24[-1]])[0]

lr_wpa = LinearRegression().fit(X_wpa, y_wpa)
lr_wpa_pred = lr_wpa.predict([X_wpa[-1]])[0]

arima_re24_pred = ARIMA(re24_real, order=(1, 0, 0)).fit().forecast(steps=1)[0]
arima_wpa_pred = ARIMA(wpa_real, order=(1, 0, 0)).fit().forecast(steps=1)[0]

ensemble_re24_pred = np.mean([rf_re24_pred, lr_re24_pred, arima_re24_pred])
ensemble_wpa_pred = np.mean([rf_wpa_pred, lr_wpa_pred, arima_wpa_pred])

ewma_re24 = pd.Series(re24_real).ewm(span=3).mean().iloc[-1]
ewma_wpa = pd.Series(wpa_real).ewm(span=3).mean().iloc[-1]

x_vals = np.arange(len(re24_real)).reshape(-1, 1)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(x_vals)
next_x_poly = poly.transform([[next_match_idx]])

poly_re24 = PolyLR().fit(X_poly, re24_real)
poly_re24_pred = poly_re24.predict(next_x_poly)[0]

poly_wpa = PolyLR().fit(X_poly, wpa_real)
poly_wpa_pred = poly_wpa.predict(next_x_poly)[0]

bootstrap_preds = []
for _ in range(1000):
    sample_idx = np.random.choice(len(X_re24), len(X_re24), replace=True)
    X_sample, y_sample = [X_re24[i] for i in sample_idx], [y_re24[i] for i in sample_idx]
    rf_bt = RandomForestRegressor(random_state=42).fit(X_sample, y_sample)
    bootstrap_preds.append(rf_bt.predict([X_re24[-1]])[0])
ci_lower, ci_upper = np.percentile(bootstrap_preds, [2.5, 97.5])

plt.figure(figsize=(14, 7))

plt.plot(re24_real, marker='o', label="Gerçek RE24", color="black", zorder=3)
plt.scatter(range(len(wpa_real)), wpa_real, marker='D', color="magenta", s=80, label="Gerçek WPA", zorder=4)

plt.axhline(rf_re24_pred, color="green", linestyle="--", alpha=0.7, label=f"RE24 RndmFrst: {rf_re24_pred:.3f}")
plt.axhline(lr_re24_pred, color="red", linestyle="--", alpha=0.7, label=f"RE24 LineerRegreyşa: {lr_re24_pred:.3f}")
plt.axhline(arima_re24_pred, color="blue", linestyle="--", alpha=0.7, label=f"RE24 ARIMA: {arima_re24_pred:.3f}")
plt.axhline(ensemble_re24_pred, color="purple", linestyle="-", linewidth=2, label=f"RE24 Ensemble: {ensemble_re24_pred:.3f}")
plt.axhline(ewma_re24, color="orange", linestyle="-.", alpha=0.7, label=f"RE24 EWMA(3): {ewma_re24:.3f}")
plt.axhline(poly_re24_pred, color="brown", linestyle="-.", alpha=0.7, label=f"RE24 PolinomTrent(2): {poly_re24_pred:.3f}")

plt.fill_between([len(re24_real)-1, next_match_idx], ci_lower, ci_upper, color="gray", alpha=0.3, label=f"RE24 %95 Confident Interval")

plt.scatter([next_match_idx], [rf_wpa_pred], color="green", marker="X", s=150, zorder=5, label=f"WPA RF: {rf_wpa_pred:.3f}")
plt.scatter([next_match_idx], [lr_wpa_pred], color="red", marker="P", s=150, zorder=5, label=f"WPA LR: {lr_wpa_pred:.3f}")
plt.scatter([next_match_idx], [arima_wpa_pred], color="blue", marker="*", s=200, zorder=5, label=f"WPA ARIMA: {arima_wpa_pred:.3f}")
plt.scatter([next_match_idx], [ensemble_wpa_pred], color="purple", marker="s", s=100, zorder=5, label=f"WPA Ensemble: {ensemble_wpa_pred:.3f}")
plt.scatter([next_match_idx], [ewma_wpa], color="orange", marker="^", s=150, zorder=5, label=f"WPA EWMA(3): {ewma_wpa:.3f}")
plt.scatter([next_match_idx], [poly_wpa_pred], color="brown", marker="v", s=150, zorder=5, label=f"WPA Polinom(2): {poly_wpa_pred:.3f}")

plt.title("Matt Olson")
plt.xlabel("Maç")
plt.ylabel("Değer (RE24 / WPA)")
plt.xticks(range(next_match_idx + 1))
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression as PolyLR
import pandas as pd

# Mevcut RE24 Verisi
re24_real = [
    -0.58, -0.09, -0.87, -0.73, -1.04, 0.45, 0.65, 0.44, -1.22, 0.22
]

# YENİ: WPA verilerini buraya kendi verilerinle güncelleyerek girebilirsin
# re24_real ile aynı maç sayısına (uzunluğa) sahip olmalıdır.
wpa_real = [
    0.05, -0.02, -0.12, -0.08, -0.15, 0.10, 0.08, 0.05, -0.10, 0.14
]

X, y = [], []
for i in range(3, len(re24_real)):
    features = [
        re24_real[i-1],
        re24_real[i-2],
        re24_real[i-3],
        np.mean(re24_real[i-3:i])
    ]
    X.append(features)
    y.append(re24_real[i])

rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)

lr = LinearRegression()
lr.fit(X, y)

import warnings
warnings.filterwarnings("ignore") # ARIMA uyarılarını gizlemek için
arima_model = ARIMA(re24_real, order=(1, 0, 0))
arima_fit = arima_model.fit()

rf_pred = rf.predict([X[-1]])[0]
lr_pred = lr.predict([X[-1]])[0]
arima_pred = arima_fit.forecast(steps=1)[0]

ensemble_pred = np.mean([rf_pred, lr_pred, arima_pred])

bootstrap_preds = []
for _ in range(1000):
    sample_idx = np.random.choice(len(X), len(X), replace=True)
    X_sample = [X[i] for i in sample_idx]
    y_sample = [y[i] for i in sample_idx]
    rf_bt = RandomForestRegressor(random_state=42)
    rf_bt.fit(X_sample, y_sample)
    bootstrap_preds.append(rf_bt.predict([X[-1]])[0])

ci_lower, ci_upper = np.percentile(bootstrap_preds, [2.5, 97.5])

ewma = pd.Series(re24_real).ewm(span=3).mean().iloc[-1]

x_vals = np.arange(len(re24_real)).reshape(-1, 1)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(x_vals)
poly_model = PolyLR()
poly_model.fit(X_poly, re24_real)
poly_pred = poly_model.predict(poly.transform([[len(re24_real)]]))[0]

# --- GRAFİK ÇİZİMİ ---
plt.figure(figsize=(12, 6))

# Orijinal RE24 Çizgisi
plt.plot(re24_real, marker='o', label="RE24", color="black", zorder=3)

# YENİ: WPA verilerini scatter (nokta) olarak ekliyoruz
# x ekseni için maç indeksleri (0, 1, 2...), y ekseni için wpa_real
plt.scatter(range(len(wpa_real)), wpa_real, marker='D', color="magenta", s=80, label="WPA", zorder=4)

# Tahmin Çizgileri
plt.axhline(rf_pred, color="green", linestyle="--", label=f"RndmFrst Tahmin: {rf_pred:.3f}")
plt.axhline(lr_pred, color="red", linestyle="--", label=f"LRggrsn Tahmin: {lr_pred:.3f}")
plt.axhline(arima_pred, color="blue", linestyle="--", label=f"ARIMA Tahmin: {arima_pred:.3f}")
plt.axhline(ensemble_pred, color="purple", linestyle="--", label=f"Ensemble(xRE24): {ensemble_pred:.3f}")

# Bootstrap Güven Aralığı
plt.fill_between(
    [len(re24_real)-1, len(re24_real)],
    ci_lower, ci_upper,
    color="gray", alpha=0.3, label=f"%95 (bootstrap) [{ci_lower:.2f}, {ci_upper:.2f}]"
)

# Diğer Tahminler
plt.axhline(ewma, color="orange", linestyle="-.", label=f"EWMA(3): {ewma:.3f}")
plt.axhline(poly_pred, color="brown", linestyle="-.", label=f"Polinom Trent AlexArnold (2. derece): {poly_pred:.3f}")

plt.title("William Contreras - RE24 ve WPA Analizi")
plt.xlabel("Maç")
plt.ylabel("Değer")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Legend'ı dışarı aldım, birbirine girmemesi için
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression as PolyLR
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

re24_real = [-0.58, -0.09, -0.87, -0.73, -1.04, 0.45, 0.65, 0.44, -1.22, 0.22]
wpa_real = [0.05, -0.02, -0.12, -0.08, -0.15, 0.10, 0.08, 0.05, -0.10, 0.14]

X_re24, y_re24 = [], []
X_wpa, y_wpa = [], []

for i in range(3, len(re24_real)):

    features_re24 = [re24_real[i-1], re24_real[i-2], re24_real[i-3], np.mean(re24_real[i-3:i])]
    X_re24.append(features_re24)
    y_re24.append(re24_real[i])


    features_wpa = [wpa_real[i-1], wpa_real[i-2], wpa_real[i-3], np.mean(wpa_real[i-3:i])]
    X_wpa.append(features_wpa)
    y_wpa.append(wpa_real[i])

next_match_idx = len(re24_real)


rf_re24 = RandomForestRegressor(random_state=42).fit(X_re24, y_re24)
rf_re24_pred = rf_re24.predict([X_re24[-1]])[0]

rf_wpa = RandomForestRegressor(random_state=42).fit(X_wpa, y_wpa)
rf_wpa_pred = rf_wpa.predict([X_wpa[-1]])[0]

lr_re24 = LinearRegression().fit(X_re24, y_re24)
lr_re24_pred = lr_re24.predict([X_re24[-1]])[0]

lr_wpa = LinearRegression().fit(X_wpa, y_wpa)
lr_wpa_pred = lr_wpa.predict([X_wpa[-1]])[0]

arima_re24_pred = ARIMA(re24_real, order=(1, 0, 0)).fit().forecast(steps=1)[0]
arima_wpa_pred = ARIMA(wpa_real, order=(1, 0, 0)).fit().forecast(steps=1)[0]

ensemble_re24_pred = np.mean([rf_re24_pred, lr_re24_pred, arima_re24_pred])
ensemble_wpa_pred = np.mean([rf_wpa_pred, lr_wpa_pred, arima_wpa_pred])

ewma_re24 = pd.Series(re24_real).ewm(span=3).mean().iloc[-1]
ewma_wpa = pd.Series(wpa_real).ewm(span=3).mean().iloc[-1]

x_vals = np.arange(len(re24_real)).reshape(-1, 1)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(x_vals)
next_x_poly = poly.transform([[next_match_idx]])

poly_re24 = PolyLR().fit(X_poly, re24_real)
poly_re24_pred = poly_re24.predict(next_x_poly)[0]

poly_wpa = PolyLR().fit(X_poly, wpa_real)
poly_wpa_pred = poly_wpa.predict(next_x_poly)[0]

bootstrap_preds = []
for _ in range(1000):
    sample_idx = np.random.choice(len(X_re24), len(X_re24), replace=True)
    X_sample, y_sample = [X_re24[i] for i in sample_idx], [y_re24[i] for i in sample_idx]
    rf_bt = RandomForestRegressor(random_state=42).fit(X_sample, y_sample)
    bootstrap_preds.append(rf_bt.predict([X_re24[-1]])[0])
ci_lower, ci_upper = np.percentile(bootstrap_preds, [2.5, 97.5])

plt.figure(figsize=(14, 7))

plt.plot(re24_real, marker='o', label="Gerçek RE24", color="black", zorder=3)
plt.scatter(range(len(wpa_real)), wpa_real, marker='D', color="magenta", s=80, label="Gerçek WPA", zorder=4)

plt.axhline(rf_re24_pred, color="green", linestyle="--", alpha=0.7, label=f"RE24 RndmFrst: {rf_re24_pred:.3f}")
plt.axhline(lr_re24_pred, color="red", linestyle="--", alpha=0.7, label=f"RE24 LineerRegreyşa: {lr_re24_pred:.3f}")
plt.axhline(arima_re24_pred, color="blue", linestyle="--", alpha=0.7, label=f"RE24 ARIMA: {arima_re24_pred:.3f}")
plt.axhline(ensemble_re24_pred, color="purple", linestyle="-", linewidth=2, label=f"RE24 Ensemble: {ensemble_re24_pred:.3f}")
plt.axhline(ewma_re24, color="orange", linestyle="-.", alpha=0.7, label=f"RE24 EWMA(3): {ewma_re24:.3f}")
plt.axhline(poly_re24_pred, color="brown", linestyle="-.", alpha=0.7, label=f"RE24 PolinomTrent(2): {poly_re24_pred:.3f}")

plt.fill_between([len(re24_real)-1, next_match_idx], ci_lower, ci_upper, color="gray", alpha=0.3, label=f"RE24 %95 Confident Interval")

plt.scatter([next_match_idx], [rf_wpa_pred], color="green", marker="X", s=150, zorder=5, label=f"WPA RF: {rf_wpa_pred:.3f}")
plt.scatter([next_match_idx], [lr_wpa_pred], color="red", marker="P", s=150, zorder=5, label=f"WPA LR: {lr_wpa_pred:.3f}")
plt.scatter([next_match_idx], [arima_wpa_pred], color="blue", marker="*", s=200, zorder=5, label=f"WPA ARIMA: {arima_wpa_pred:.3f}")
plt.scatter([next_match_idx], [ensemble_wpa_pred], color="purple", marker="s", s=100, zorder=5, label=f"WPA Ensemble: {ensemble_wpa_pred:.3f}")
plt.scatter([next_match_idx], [ewma_wpa], color="orange", marker="^", s=150, zorder=5, label=f"WPA EWMA(3): {ewma_wpa:.3f}")
plt.scatter([next_match_idx], [poly_wpa_pred], color="brown", marker="v", s=150, zorder=5, label=f"WPA Polinom(2): {poly_wpa_pred:.3f}")

plt.title("William Contreras")
plt.xlabel("Maç")
plt.ylabel("Değer (RE24 / WPA)")
plt.xticks(range(next_match_idx + 1))
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
#Tahmin ve NetRE24 tablosu
import matplotlib.pyplot as plt
import numpy as np

categories = ['Profar', 'Matty', 'Ronald', 'Riley', 'MHll', 'Ozuna', 'Drake', 'Ozzie', 'Nick', 'Eli', 'Sean']
expected_values = [0.2, 1.1, 0.2, -0.2, 0.6, -0.1, 1.7, -0.3, -0.5, 0.4, -0.3]
net_values = [0.2, 0.47, -0.13, -0.91, 0.66, 2.35, -1.25, 0.27, -0.06, 1.26, -0.44]
Best_RE24_Number = [3, 10, 3, 10, 7, 6, 7, 6, 0, 1, 8]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
plt.style.use('seaborn-v0_8-darkgrid')

bars1 = ax.bar(x - width/2, expected_values, width, label='Expected RE24', color='#4daf4a', edgecolor='black')
bars2 = ax.bar(x + width/2, net_values, width, label='Net RE24', color='#800080', edgecolor='black')

scatter = ax.scatter(x, Best_RE24_Number, color='red', label='BestRE24Number', zorder=5, marker='o', s=80, edgecolors='black')

# Başlık ve etiketler
ax.set_title('Expected vs. Net RE24 with Impact Score\n29.07.25 vs. Royals', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Batters', fontsize=14)
ax.set_ylabel('RE24 Value / Score', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=12)

ax.legend(fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Bar üstü etiketler
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5 if height >= 0 else -15),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, color='black')

add_labels(bars1)
add_labels(bars2)

# Scatter noktaları için etiketler (opsiyonel)
for i, score in enumerate(Best_RE24_Number):
    ax.annotate(f'{score:.1f}', (x[i], score), textcoords="offset points",
                xytext=(0, 8), ha='center', fontsize=9, color='darkred')

# Y ekseni sınırları
all_values = expected_values + net_values + Best_RE24_Number
y_min = min(all_values) - 3
y_max = max(all_values) + 3
ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

re24_real = [
    1.05, 0.62, 0.21, -0.01, 1.16, -0.1, -0.55, 2.23, -0.98, 0.04,
    -0.17, -0.11, -0.24, -0.74, -0.40, 1.30, 1.29, 1.69, 1.38, 2.51, 0.53, 1.39, 0.41, 0.34, 0.06, 2.05, 0.61, 0.96, -0.68, 1.73, -0.83, -0.24, 0.18
]

X, y = [], []
for i in range(3, len(re24_real)):
    features = [
        re24_real[i-1],
        re24_real[i-2],
        re24_real[i-3],
        np.mean(re24_real[i-3:i])
    ]
    X.append(features)
    y.append(re24_real[i])

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

simulated_re24s = re24_real.copy()
for i in range(len(re24_real), 162):
    last3 = simulated_re24s[-3:]
    test_input = [
        last3[2], last3[1], last3[0],
        np.mean(last3)
    ]
    predicted_value = model.predict([test_input])[0]
    noise = np.random.normal(0, 0.8)
    simulated_value = predicted_value + noise
    simulated_re24s.append(simulated_value)

results = []
for i in range(33, 111):
    value = simulated_re24s[i-1]
    durum = "✅" if value > 0 else "❌"
    results.append((i, value, durum))

# Doğru uzunlukta x-y eşleşmesi
match_numbers = [i for i, _, _ in results]
re24_values = [v for _, v, _ in results]

# Pozitif/negatif sayımı
positive_count = sum(1 for v in re24_values if v > 0)
negative_count = sum(1 for v in re24_values if v <= 0)

plt.figure(figsize=(12, 6))
plt.scatter(
    match_numbers, re24_values,
    c=['green' if v > 0 else 'red' for v in re24_values],
    alpha=0.7
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title(f"Ronald 2025 sezonu RE24 Tahmini\n0+: {positive_count} maç / 0-: {negative_count} maç")
plt.xlabel("Maç")
plt.ylabel("RE24 Tahmini")
plt.grid(True)
plt.tight_layout()
plt.show()